# MLOps Altyapısı — VisionLab

Bu notebook, **model eğitimi bittikten sonra** eklenen tüm MLOps bileşenlerini
belgeliyor: veri versiyonlama, deney takibi/model kaydı, self-describing
checkpoint'ler, tahmin servisi, ve kullanıcı geri bildirimiyle beslenen
sürekli öğrenme döngüsü ("flywheel"). Mantar/çiçek sınıflandırma
araştırması `08_mushroom_confusion_diagnosis.ipynb`'da; burası tamamen
domain-agnostic altyapı notları.

## İçindekiler
1. DVC — Veri Versiyonlama
2. MLflow — Deney Takibi + Model Registry
3. Self-Describing Checkpoint
4. FastAPI Serving Katmanı
5. Flywheel — Sürekli Öğrenme Döngüsü
6. Şu Anki Olgunluk Durumu ve Eksikler


## 1) DVC — Veri Versiyonlama

**Ne sorunu çözüyor:** "Bu model hangi tam veri durumuyla eğitildi?" sorusuna
kesin cevap. Git büyük binary dosyaları (13GB'lık görsel verisi) tutamaz;
DVC bunun için var — içerik hash'ini git'e koyar, gerçek veriyi ayrı bir
cache'te tutar.

**Kurulum:**
- `dvc init` — repoyu DVC'ye bağladı.
- `dvc.yaml` — `preprocess_mushroom` / `preprocess_flower` pipeline
  adımları, mevcut `src/data/loaders.py` mantığını sarmalayıp
  `datasets/processed/{mushroom,flower}/{train,val,test}.csv` üretiyor.
- `datasets/extracted/mushroom` (12GB) ve `datasets/extracted/flower_data`
  (348MB) `dvc add` ile versiyonlandı — `.dvc` pointer dosyaları git'te,
  gerçek görseller değil.

**Çıkan sorunlar ve çözümleri:**
- `datasets/raw/*.zip` dosyalarının hâlâ bozuk placeholder (786KB stub)
  olduğu ortaya çıktı — gerçek veri Kaggle API'den doğrudan `extracted/`'a
  inmişti. Bu yüzden "ham veri → extract" aşaması kurulmadı, `extracted/`
  doğrudan taban veri olarak alındı.
- `dvc push` (uzak depoya gönderme) **disk kotası dolu** hatası verdi — `sum`
  proje alanının paylaşılan kotası zaten sınırdaydı. `.dvc/cache`'i
  hardlink'e çevirerek (`dvc checkout --relink`) 12GB'lık gereksiz ikinci
  kopyayı geri kazandık. **Sonuç: yerel versiyonlama tam çalışıyor, uzak
  depo (push) için TRUBA'dan ek kota gerekiyor — henüz istenmedi.**


## 2) MLflow — Deney Takibi + Model Registry

**Ne sorunu çözüyor:** W&B "hangi deney en iyi sonucu verdi" sorusuna cevap
veriyor (grafikler, karşılaştırma). Ama "şu an production'da hangi model
var" sorusuna cevap vermiyor — bunun için bir **Model Registry** gerekiyor.

**Kurulum:** `src/training/tracking.py`'a W&B'nin **yanına paralel** (yerine
değil) MLflow fonksiyonları eklendi. `train_baseline.py` ve
`train_stage2_finetune.py` her eğitimde her ikisine de log atıyor. En iyi
checkpoint MLflow **Model Registry**'ye kaydediliyor.

**Modern "production" mekanizması:** MLflow'un eski "stage" sistemi
(None/Staging/Production/Archived) artık deprecated — bunun yerine
**alias** kullanılıyor (`client.set_registered_model_alias(name,
"production", version)`). Serving katmanı `models:/<name>@production`
şeklinde okuyor.

**Çıkan sorunlar ve çözümleri:**
- MLflow 3.x, eski dosya-tabanlı `mlruns/` backend'ini artık desteklemiyor
  ("maintenance mode" hatası) → SQLite backend'ine (`sqlite:///mlflow.db`)
  geçirdik.
- `mlflow.pytorch.log_model`'in yeni varsayılanı ('pt2' formatı)
  `input_example` zorunlu kılıyor ve modelin traceable olmasını
  gerektiriyor — özel mimarilerimiz (özellikle tuple dönen hiyerarşik
  model) için garanti değil → klasik `pickle` serileştirmeye sabitlendi.
- `log_model`'in kendi `tags=` parametresi, registry'deki model
  **versiyonuna değil**, run/artifact metadata'sına yazılıyormuş (serving
  katmanını kurarken `KeyError: 'label_map'` ile yakalandı) →
  `client.set_model_version_tag()` ile açıkça düzeltildi.


## 3) Self-Describing Checkpoint

**Ne sorunu çözüyor:** 6 ay sonra bulunan bir `.pt` dosyasının hangi
kod/config/veri ile üretildiğini tahmin etmek zorunda kalmamak.

**`src/utils/run_metadata.py`** her eğitimde şunları topluyor:
- git commit hash + branch + "dirty" durumu
- config dosyasının **içerik hash'i** (dosya adı değil — aynı isimde farklı
  içerik olursa yakalar)
- DVC veri versiyon hash'i (`.dvc` pointer dosyasından okunuyor)

**`save_checkpoint`**'e eklenen alanlar:
- `run_metadata` — yukarıdaki üçlü
- `label_map` — sınıf isim↔indeks eşlemesi, dosya yoluna değil doğrudan
  checkpoint'e gömülü
- `image_size` — inference'ta hangi preprocessing'in uygulanacağı

Bu üç alan aynı zamanda MLflow model versiyonuna **tag** olarak da
yazılıyor — serving katmanı checkpoint dosyasına hiç dokunmadan, sadece
registry'den okuyarak çalışabiliyor.


## 4) FastAPI Serving Katmanı

**Tasarım prensibi:** Endpoint yapısı veri setine göre değil, göreve göre
genel. Yeni bir domain (örn. PlantVillage) eklendiğinde API kodunda hiçbir
değişiklik gerekmiyor — sadece registry'ye yeni bir model kaydedip
`production` alias'ını ona taşımak yeterli.

**Modüller:**
- `src/inference/registry.py` — `production` alias'lı her modeli, tag'lerden
  (label_map, image_size, dataset_type) okuyarak yükler.
- `src/inference/predictor.py` — tek, domain-agnostic `predict()`.
- `src/deployment/api.py` — `POST /predict`, `GET /models`, `GET /health`
  (+ flywheel entegrasyonu, bkz. bölüm 5).
- `scripts/register_model.py` — bu entegrasyondan önce eğitilmiş
  checkpoint'leri (mantar Stage 1, çiçek Run 3) registry'ye geriye dönük
  kaydetmek için.
- `scripts/serve.sbatch` — TRUBA'da uzun süreli çalıştırmak için (manuel,
  otomatik başlatılmıyor).

**Uçtan uca doğrulandı:** Mantar modeliyle gerçek bir validation görseli
(`Amanita muscaria`) → doğru tahmin, %99.99 güven; çiçek modeliyle
(`pink primrose`) → doğru tahmin, %99.99 güven — **aynı kod yolu**, hiçbir
domain-özel dallanma olmadan.


## 5) Flywheel — Sürekli Öğrenme Döngüsü

**Fikir:** Kullanıcı fotoğraf yükler → model tahmin eder → tahmin
biriktirilir → bir insan doğrular/düzeltir → doğrulanmış veri belli bir
eşiğe ulaşınca modele eklenir → yeniden eğitim → **sadece mevcut
production'dan iyiyse** yeni model production'a çıkar.

**Kritik prensip:** Bir tahmin, etiket değildir. Model kendi tahminiyle
kendini eğitemez — bu "confirmation bias" yaratır, hatalar katlanarak
büyür. Bu yüzden `true_label` alanı **sadece** `POST /feedback` çağrısıyla
doluyor; hiçbir tahmin insan onayı olmadan eğitim verisi haline gelmiyor.

**Bileşenler:**

| Dosya | Görev |
|---|---|
| `src/flywheel/store.py` | SQLite (`flywheel.db`) — her `/predict` çağrısını, ve varsa doğrulanmış etiketini kaydeder |
| `src/deployment/api.py` (`POST /feedback`) | İnsan doğrulaması — tahmini onaylar/düzeltir |
| `src/deployment/api.py` (`GET /flywheel/stats`) | Kaba izleme — toplam/doğrulanmış/ortalama güven |
| `src/flywheel/export.py` | Doğrulanmış+kullanılmamış kayıtları, `datasets/extracted/<dataset>_flywheel/` altında DVC-track edilebilir bir snapshot'a çevirir |
| `scripts/check_flywheel_threshold.py` | Doğrulanmış veri sayısı eşiği geçtiyse yeniden eğitimi tetikler (otomatik zamanlanmıyor — elle ya da crontab ile çalıştırılmalı) |
| `scripts/flywheel_retrain.sbatch` | export → `dvc add` → `train_baseline.py --extra-train-csv` → `promote_if_better.py` — tek SLURM job'ında zincirlenmiş |
| `scripts/promote_if_better.py` | **Terfi kapısı** — adayın macro-F1'i mevcut production'ı geçmiyorsa, `production` alias'ı değişmiyor |

**Bilinen sınırlama:** `export.py`, `true_label`'ı mevcut `label_map`'te
olmayan (yani modelin hiç görmediği yeni bir tür) kayıtları atlıyor. Gerçek
yeni bir tür eklemek, çıkış katmanını genişletmeyi gerektirir — bu ayrı,
bilinçli bir mimari değişiklik, flywheel'in sessizce yapacağı bir şey değil.

**Uçtan uca test edildi:** `/predict` → `prediction_id` döndü → `/feedback`
ile doğrulandı → `/flywheel/stats` doğru sayıyı gösterdi → `export_flywheel_snapshot()`
doğru CSV'yi üretti ve kaydı `used_in_training` işaretledi → eşik kontrolü
bunu doğru yansıttı. Gerçek bir `flywheel_retrain.sbatch` koşusu (saatler
süren bir eğitim gerektirdiği için) bu oturumda çalıştırılmadı — parçalar
ayrı ayrı doğrulandı.


## 6) Şu Anki Olgunluk Durumu ve Eksikler

| Bileşen | Durum |
|---|---|
| Veri versiyonlama (DVC) | ✅ Yerel tam çalışıyor, ❌ uzak depo (kota) |
| Deney takibi + Model Registry (MLflow) | ✅ |
| Self-describing checkpoint | ✅ |
| Serving (FastAPI) | ✅ |
| Tahmin loglama | ✅ |
| İnsan doğrulama (feedback) | ✅ (API var, gerçek bir doğrulama arayüzü/süreci yok) |
| Veri birikimi → eğitim setine katma | ✅ |
| Otomatik yeniden eğitim tetikleyicisi | ✅ (script var, **zamanlanmadı** — crontab elle kurulmalı) |
| Otomatik karşılaştırma/terfi kapısı | ✅ |
| İzleme/drift tespiti | ⚠️ Sadece kaba istatistik (`/flywheel/stats`) — gerçek drift/anomali tespiti yok |
| CI/CD | ❌ Yok |

**Bundan sonraki gerçek kararlar (kod değil, ürün/süreç kararı gerektiren):**
1. Doğrulama sürecini kim/nasıl yapacak? (`/feedback`'i kim çağıracak —
   tek bir uzman mı, kullanıcı topluluğu mu?)
2. `check_flywheel_threshold.py` ne sıklıkla çalışacak — crontab mı,
   elle mi?
3. DVC uzak depo kotası TRUBA'dan istenecek mi?
4. Eşik değeri (`--threshold`, şu an varsayılan 100) gerçek kullanım
   hacmine göre ayarlanmalı.


## 7) Flywheel Politikası — Somutlaştırılan 3 Karar

Bölüm 6'da "kod değil, ürün/süreç kararı" olarak listelenen üç madde artık
koda yansıtıldı ve **[`RETRAINING_POLICY.md`](../RETRAINING_POLICY.md)**'de
belgelendi:

1. **Doğrulama kuyruğu filtresi** — sadece `confidence < 0.70` olan
   tahminler `GET /flywheel/pending`'e düşüyor (`needs_verification` kolonu,
   `log_prediction()` içinde hesaplanıyor). Yüksek güvenli tahminler
   loglanıyor ama kuyruğa hiç girmiyor — `/feedback` yine de herhangi bir
   `prediction_id` için elle çağrılabilir, sadece otomatik kuyruğa düşmüyor.
2. **Zaman + sayı bazlı tetikleme** — `check_flywheel_threshold.py` artık
   `verified_count >= 500 OR days_since_last_check >= 30` kontrolü yapıyor
   (`src/flywheel/store.py`'daki `retrain_triggers` tablosu, en son ne zaman
   tetiklendiğini takip ediyor).
3. **Terfi kapısı** — bu zaten `flywheel_retrain.sbatch`'in son adımıydı
   (`promote_if_better.py`), `register_model.py --production`'a hiç
   çağrı yapılmıyordu. Gözden geçirilip doğrulandı, değişiklik gerekmedi.

**Doğrulama:** `src/flywheel/store.py` fonksiyonları doğrudan (yüksek/düşük
güvenli tahmin ayrımı, `days_since_last_trigger`), `check_flywheel_threshold.py`
üç senaryoda (eşik altı → atla, sayı eşiği aşıldı → tetikle, hiç
tetiklenmemiş → zaman koşuluyla tetikle), ve gerçek bir `/predict` çağrısının
(%99.99 güven) `/flywheel/pending`'e düşmediği canlı servis üzerinden
doğrulandı.
